# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [21]:
import re
import tarfile
import pandas as pd

from pathlib import Path
from collections import Counter
from math import log

def preprocesar(texto):
    return re.findall(r"[^\W\d_]+", texto.lower())
archivo_tar = Path(
    r"C:\Users\ASUS\Documents\gutenberg_grande\txt-files.tar"
)

if not archivo_tar.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {archivo_tar}")
limite_documentos = 1000
filas = []
ids_seleccionados = set()

print("Cargando documentos del corpus grande...", flush=True)

with tarfile.open(archivo_tar, "r:") as corpus_tar:

    for archivo in corpus_tar:
        coincidencia = re.fullmatch(
            r"cache/epub/(\d+)/pg(\d+)\.txt",
            archivo.name,
            flags=re.IGNORECASE
        )

        if not archivo.isfile() or archivo.size == 0:
            continue

        if coincidencia is None:
            continue

        id_carpeta = coincidencia.group(1)
        id_archivo = coincidencia.group(2)
        if id_carpeta != id_archivo:
            continue

        if id_carpeta in ids_seleccionados:
            continue
        with corpus_tar.extractfile(archivo) as libro:
            contenido = libro.read()

        try:
            texto = contenido.decode("utf-8-sig")
        except UnicodeDecodeError:
            continue

        tokens = preprocesar(texto)

        if not tokens:
            continue
        encabezado = re.search(
            r"^Title:\s*(.+)$",
            texto[:10000],
            flags=re.IGNORECASE | re.MULTILINE
        )

        if encabezado:
            titulo = encabezado.group(1).strip()
        else:
            titulo = f"Título no encontrado (ID {id_carpeta})"
        filas.append({
            "Identificador de documento": id_carpeta,
            "Título": titulo,
            "Texto": texto,
            "tokens": tokens
        })

        ids_seleccionados.add(id_carpeta)

        if len(filas) % 100 == 0:
            print(f"Documentos cargados: {len(filas)}", flush=True)

        if len(filas) == limite_documentos:
            break
corpus = pd.DataFrame(filas)

if corpus.empty:
    raise ValueError("No se pudo cargar ningún documento del TAR.")
N = len(corpus)
df_corpus = Counter()

for tokens in corpus["tokens"]:
    df_corpus.update(set(tokens))
idf_corpus = {
    termino: log(N / df) + 1
    for termino, df in df_corpus.items()
}
estadisticas_terminos = pd.DataFrame([
    {
        "Término": termino,
        "DF": df,
        "IDF": idf_corpus[termino]
    }
    for termino, df in df_corpus.items()
])


print(f"\nDocumentos cargados: {N}")
print(f"Términos distintos: {len(df_corpus):,}")

display(corpus.head())
display(estadisticas_terminos.head())

Cargando documentos del corpus grande...
Documentos cargados: 100
Documentos cargados: 200
Documentos cargados: 300
Documentos cargados: 400
Documentos cargados: 500
Documentos cargados: 600
Documentos cargados: 700
Documentos cargados: 800
Documentos cargados: 900
Documentos cargados: 1000

Documentos cargados: 1000
Términos distintos: 647,766


,Identificador de documento,Título,Texto,tokens
0,10000,The Magna Carta,The Project Gutenberg eBook of The Magna Carta...,"[the, project, gutenberg, ebook, of, the, magn..."
1,10001,Apocolocyntosis,The Project Gutenberg eBook of Apocolocyntosis...,"[the, project, gutenberg, ebook, of, apocolocy..."
2,10002,The House on the Borderland,The Project Gutenberg eBook of The House on th...,"[the, project, gutenberg, ebook, of, the, hous..."
3,10003,"My First Years as a Frenchwoman, 1876-1879",The Project Gutenberg eBook of My First Years ...,"[the, project, gutenberg, ebook, of, my, first..."
4,10004,The Warriors,The Project Gutenberg eBook of The Warriors\r\...,"[the, project, gutenberg, ebook, of, the, warr..."


,Término,DF,IDF
0,in,1000,1.000000
1,attendants,255,2.366492
2,was,1000,1.000000
3,gives,734,1.309246
4,customary,312,2.164752


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [22]:
import pandas as pd
from IPython.display import display

consultas = [
    ("the", "Comprobar qué recupera una consulta con un artículo muy frecuente."),
    ("love", "Encontrar libros que hablan del amor."),
    ("war peace", "Encontrar textos sobre la guerra y la paz."),
    ("queequeg", "Encontrar el libro que menciona al personaje Queequeg."),
    ("elinor dashwood", "Localizar textos en los que se menciona a Elinor Dashwood."),
    ("whale ship captain", "Encontrar relatos sobre barcos, capitanes y ballenas."),
    ("king queen royal court", "Encontrar pasajes sobre la realeza y la corte."),
    ("faith prayer god heaven salvation", "Encontrar textos sobre la fe y la salvación."),
]

N = len(corpus)

tabla_consultas = pd.DataFrame([
    {
        "Consulta": consulta,
        "Qué busca el usuario": objetivo,
        "Longitud": len(preprocesar(consulta)),
        "DF por término": ", ".join(
            f"{termino}: {df_corpus[termino]}/{N}"
            for termino in preprocesar(consulta)
        ),
    }
    for consulta, objetivo in consultas
])

display(tabla_consultas)

,Consulta,Qué busca el usuario,Longitud,DF por término
0,the,Comprobar qué recupera una consulta con un art...,1,the: 1000/1000
1,love,Encontrar libros que hablan del amor.,1,love: 824/1000
2,war peace,Encontrar textos sobre la guerra y la paz.,2,"war: 747/1000, peace: 763/1000"
3,queequeg,Encontrar el libro que menciona al personaje Q...,1,queequeg: 0/1000
4,elinor dashwood,Localizar textos en los que se menciona a Elin...,2,"elinor: 13/1000, dashwood: 3/1000"
5,whale ship captain,"Encontrar relatos sobre barcos, capitanes y ba...",3,"whale: 142/1000, ship: 623/1000, captain: 550/..."
6,king queen royal court,Encontrar pasajes sobre la realeza y la corte.,4,"king: 723/1000, queen: 579/1000, royal: 574/10..."
7,faith prayer god heaven salvation,Encontrar textos sobre la fe y la salvación.,5,"faith: 667/1000, prayer: 497/1000, god: 781/10..."


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [29]:
import pandas as pd

from itertools import combinations, zip_longest
from collections import Counter
from IPython.display import display
titulos_por_id = dict(zip(
    corpus["Identificador de documento"].astype(str),
    corpus["Título"]
))

tokens_por_id = dict(zip(
    corpus["Identificador de documento"].astype(str),
    corpus["tokens"]
))
comparaciones = {}
for consulta, objetivo in consultas:

    ranking_tfidf, ranking_bm25 = comparar_consulta(consulta)

    comparaciones[consulta] = (ranking_tfidf, ranking_bm25)

    filas = []

    for posicion, (doc_tfidf, doc_bm25) in enumerate(
        zip_longest(ranking_tfidf, ranking_bm25),
        start=1
    ):

        if doc_tfidf is not None:
            nombre_tfidf = (
                f"{doc_tfidf['doc_id']} | {doc_tfidf['Título']}"
            )
            coseno = doc_tfidf["Score"]
        else:
            nombre_tfidf = "—"
            coseno = None

        if doc_bm25 is not None:
            nombre_bm25 = (
                f"{doc_bm25['doc_id']} | {doc_bm25['Título']}"
            )
            score_bm25 = doc_bm25["Score"]
        else:
            nombre_bm25 = "—"
            score_bm25 = None

        filas.append({
            "Posición": posicion,
            "ID y título TF-IDF": nombre_tfidf,
            "Coseno": coseno,
            "ID y título BM25": nombre_bm25,
            "Score BM25": score_bm25
        })

    print(f"\nConsulta: {consulta}")
    print(f"Objetivo: {objetivo}")

    tabla_ranking = pd.DataFrame(
        filas,
        columns=[
            "Posición",
            "ID y título TF-IDF",
            "Coseno",
            "ID y título BM25",
            "Score BM25"
        ]
    )

    display(tabla_ranking)
    ids_tfidf = {doc["doc_id"] for doc in ranking_tfidf}
    ids_bm25 = {doc["doc_id"] for doc in ranking_bm25}

    print(f"Documentos compartidos: {len(ids_tfidf & ids_bm25)}")
    print(f"Documentos en TF-IDF: {len(ranking_tfidf)}")
    print(f"Documentos en BM25: {len(ranking_bm25)}")

    if ranking_tfidf:
        print(f"Primer documento TF-IDF: {ranking_tfidf[0]['doc_id']}")
    else:
        print("TF-IDF: sin resultados positivos")

    if ranking_bm25:
        print(f"Primer documento BM25: {ranking_bm25[0]['doc_id']}")
    else:
        print("BM25: sin resultados positivos")
ejemplo = None

for consulta, (ranking_tfidf, ranking_bm25) in comparaciones.items():

    posiciones_tfidf = {
        doc["doc_id"]: posicion
        for posicion, doc in enumerate(ranking_tfidf, start=1)
    }

    posiciones_bm25 = {
        doc["doc_id"]: posicion
        for posicion, doc in enumerate(ranking_bm25, start=1)
    }

    comunes = posiciones_tfidf.keys() & posiciones_bm25.keys()

    for doc_a, doc_b in combinations(sorted(comunes), 2):

        distancia_tfidf = abs(
            posiciones_tfidf[doc_a] - posiciones_tfidf[doc_b]
        )

        distancia_bm25 = abs(
            posiciones_bm25[doc_a] - posiciones_bm25[doc_b]
        )

        ambos_cambian = (
            posiciones_tfidf[doc_a] != posiciones_bm25[doc_a]
            and posiciones_tfidf[doc_b] != posiciones_bm25[doc_b]
        )

        if ambos_cambian or distancia_tfidf != distancia_bm25:
            ejemplo = (consulta, doc_a, doc_b)
            break

    if ejemplo is not None:
        break
if ejemplo is None:

    print(
        "\nNo se encontró un par compartido con el cambio "
        "de posiciones solicitado."
    )

else:

    consulta, doc_a, doc_b = ejemplo
    ranking_tfidf, ranking_bm25 = comparaciones[consulta]

    posiciones_tfidf = {
        doc["doc_id"]: posicion
        for posicion, doc in enumerate(ranking_tfidf, start=1)
    }

    posiciones_bm25 = {
        doc["doc_id"]: posicion
        for posicion, doc in enumerate(ranking_bm25, start=1)
    }

    scores_tfidf = {
        doc["doc_id"]: doc["Score"]
        for doc in ranking_tfidf
    }

    scores_bm25 = {
        doc["doc_id"]: doc["Score"]
        for doc in ranking_bm25
    }

    print(f"\nConsulta elegida: {consulta}")
    print(f"Longitud media del corpus: {longitud_media:.2f} tokens")
    tabla_ejemplo = pd.DataFrame([
        {
            "ID": doc_id,
            "Título": titulos_por_id[doc_id],
            "Posición TF-IDF": posiciones_tfidf[doc_id],
            "Coseno": scores_tfidf[doc_id],
            "Posición BM25": posiciones_bm25[doc_id],
            "Score BM25": scores_bm25[doc_id]
        }
        for doc_id in [doc_a, doc_b]
    ])

    display(tabla_ejemplo)
    terminos = list(dict.fromkeys(preprocesar(consulta)))
    filas_aportes = []

    for doc_id in [doc_a, doc_b]:

        tokens = tokens_por_id[doc_id]
        frecuencias = Counter(tokens)
        longitud = len(tokens)

        ajuste = k1 * (
            1 - b + b * longitud / longitud_media
        )

        for termino in terminos:

            tf = frecuencias[termino]
            idf = idf_bm25.get(termino, 0)

            aporte = idf * tf * (k1 + 1) / (tf + ajuste)

            filas_aportes.append({
                "ID": doc_id,
                "Longitud": longitud,
                "Término": termino,
                "TF": tf,
                "DF": df_corpus.get(termino, 0),
                "Aporte BM25": aporte
            })

    tabla_aportes = pd.DataFrame(filas_aportes)

    display(tabla_aportes)


Consulta: the
Objetivo: Comprobar qué recupera una consulta con un artículo muy frecuente.


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,"10702 | The History of Rome, Book II",0.785267,10706 | The History of Rome (Volumes 1-5),0.001099
1,2,10908 | Manual for Noncommissioned Officers an...,0.784042,10998 | A Catechism of the Steam Engine,0.001099
2,3,10706 | The History of Rome (Volumes 1-5),0.778681,"10704 | The History of Rome, Book IV",0.001099
3,4,"10701 | The History of Rome, Book I",0.778043,"10705 | The History of Rome, Book V",0.001099
4,5,"10704 | The History of Rome, Book IV",0.777579,"10703 | The History of Rome, Book III",0.001099
5,6,"10703 | The History of Rome, Book III",0.765400,"10702 | The History of Rome, Book II",0.001099
6,7,"10705 | The History of Rome, Book V",0.763329,11034 | A Compilation of the Messages and Pape...,0.001099
7,8,10453 | A Practical Physiology: A Text-Book fo...,0.761150,"10701 | The History of Rome, Book I",0.001099
8,9,"10726 | Outlines of Lessons in Botany, Part I;...",0.761067,10919 | A Compilation of the Messages and Pape...,0.001099
9,10,10795 | Yorkshire—Coast and Moorland Scenes,0.760153,10453 | A Practical Physiology: A Text-Book fo...,0.001099


Documentos compartidos: 7
Documentos en TF-IDF: 10
Documentos en BM25: 10
Primer documento TF-IDF: 10702
Primer documento BM25: 10706

Consulta: love
Objetivo: Encontrar libros que hablan del amor.


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,1105 | The Sonnets,0.127259,1105 | The Sonnets,0.425527
1,2,1041 | Shakespeare's Sonnets,0.127129,1041 | Shakespeare's Sonnets,0.425518
2,3,10913 | English Poems,0.087270,"10665 | Life, letters, and Epicurean philosoph...",0.425360
3,4,1045 | Venus and Adonis,0.085102,10740 | The Way of Peace,0.425173
4,5,10740 | The Way of Peace,0.074478,10314 | The Sonnets of Michael Angelo Buonarro...,0.425073
5,6,10314 | The Sonnets of Michael Angelo Buonarro...,0.069199,"10734 | A Collection of Old English Plays, Vol...",0.424947
6,7,"10910 | Robert Louis Stevenson, an Elegy; and ...",0.066043,"10039 | The Works of Aphra Behn, Volume III",0.424864
7,8,10495 | Under King Constantine,0.061884,10913 | English Poems,0.424835
8,9,10518 | Poems,0.058022,10518 | Poems,0.424705
9,10,"10665 | Life, letters, and Epicurean philosoph...",0.056797,10949 | The Romance of Zion Chapel [3d ed.],0.424648


Documentos compartidos: 7
Documentos en TF-IDF: 10
Documentos en BM25: 10
Primer documento TF-IDF: 1105
Primer documento BM25: 1105

Consulta: war peace
Objetivo: Encontrar textos sobre la guerra y la paz.


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,10090 | Peaceless Europe,0.075837,10090 | Peaceless Europe,1.233585
1,2,10333 | The European Anarchy,0.068712,10291 | In the Fourth Year: Anticipations of a...,1.230663
2,3,10159 | England and the War,0.060171,10333 | The European Anarchy,1.230495
3,4,10291 | In the Fourth Year: Anticipations of a...,0.053182,10990 | Selected Speeches on British Foreign P...,1.229851
4,5,"10166 | What Germany Thinks; Or, The War as Ge...",0.052530,10444 | The Peace Negotiations: A Personal Nar...,1.229652
5,6,10097 | The Healing of Nations and the Hidden ...,0.049283,"10166 | What Germany Thinks; Or, The War as Ge...",1.229012
6,7,"10507 | Mary, Erzählung",0.046731,"10809 | Why We Are at War (2nd Edition, revised)",1.228638
7,8,"10809 | Why We Are at War (2nd Edition, revised)",0.044504,10895 | A Compilation of the Messages and Pape...,1.227454
8,9,10078 | Plain Words from America: A Letter to ...,0.042459,10159 | England and the War,1.225839
9,10,10694 | Sea-Power and Other Studies,0.042244,"10703 | The History of Rome, Book III",1.225806


... (9 outputs left out)

,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,10325 | The Gospel of the Pentateuch: A Set of...,0.072472,10801 | The Literary Remains of Samuel Taylor ...,6.176543
1,2,"1046 | God, the Invisible King",0.068785,10129 | Excellent Women,6.142588
2,3,10116 | All Saints' Day and Other Sermons,0.063572,10831 | The Village in the Mountains; Conversi...,6.139937
3,4,10326 | David: Five Sermons,0.044132,"1046 | God, the Invisible King",6.089542
4,5,10058 | The Divine Office: A Study of the Roma...,0.035052,10900 | The King James Bible,6.085968
5,6,10831 | The Village in the Mountains; Conversi...,0.034520,10 | The King James Version of the Bible,6.085793
6,7,10314 | The Sonnets of Michael Angelo Buonarro...,0.033235,1037 | The Life of John Bunyan,6.082362
7,8,10801 | The Literary Remains of Samuel Taylor ...,0.032455,10058 | The Divine Office: A Study of the Roma...,6.056716
8,9,10553 | The first New Testament printed in Eng...,0.031568,10642 | Towards the Great Peace,5.981025
9,10,10375 | England's Antiphon,0.031234,"10369 | Memoir and Diary of John Yeardley, Min...",5.962842


Documentos compartidos: 4
Documentos en TF-IDF: 10
Documentos en BM25: 10
Primer documento TF-IDF: 10325
Primer documento BM25: 10801

Consulta elegida: the
Longitud media del corpus: 72554.36 tokens


,ID,Título,Posición TF-IDF,Coseno,Posición BM25,Score BM25
0,10453,A Practical Physiology: A Text-Book for Higher...,8,0.761150,10,0.001099
1,10701,"The History of Rome, Book I",4,0.778043,8,0.001099


,ID,Longitud,Término,TF,DF,Aporte BM25
0,10453,143702,the,14598,1000,0.001099
1,10701,111871,the,12242,1000,0.001099


**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [30]:
from collections import Counter

import pandas as pd
from IPython.display import display

consulta = "whale ship captain"
terminos = list(dict.fromkeys(preprocesar(consulta)))

configuraciones = [
    ("Base", 1.2, 0.75),
    ("Sin ajuste por longitud", 1.2, 0.00),
    ("Mayor efecto de las repeticiones", 2.0, 0.75),
]
frecuencias = {}
longitudes = {}

for doc_id, tokens in tokens_por_id.items():
    conteo = Counter(tokens)
    if any(conteo[t] > 0 for t in terminos):
        frecuencias[doc_id] = conteo
        longitudes[doc_id] = len(tokens)
resultados = {}

for nombre, k1_prueba, b_prueba in configuraciones:
    aportes = {}
    scores = {}

    for doc_id, conteo in frecuencias.items():
        longitud = longitudes[doc_id]
        ajuste = k1_prueba * (1 - b_prueba + b_prueba * longitud / longitud_media)

        aportes[doc_id] = {}
        for t in terminos:
            tf = conteo[t]
            aportes[doc_id][t] = (
                idf_bm25.get(t, 0) * tf * (k1_prueba + 1) / (tf + ajuste)
            )

        scores[doc_id] = sum(aportes[doc_id].values())

    orden = sorted(scores, key=lambda doc_id: (-scores[doc_id], doc_id))
    posiciones = {doc_id: pos for pos, doc_id in enumerate(orden, start=1)}
    resultados[nombre] = (posiciones, scores, aportes)
mas_corto = min(longitudes, key=longitudes.get)
mas_largo = max(longitudes, key=longitudes.get)
primero_base = min(resultados["Base"][0], key=resultados["Base"][0].get)
elegidos = list(dict.fromkeys([mas_corto, mas_largo, primero_base]))
filas_posiciones = []

for doc_id in elegidos:
    fila = {"doc_id": doc_id, "|d|": longitudes[doc_id]}

    for nombre, (posiciones, scores, _) in resultados.items():
        fila[f"Posición · {nombre}"] = posiciones[doc_id]
        fila[f"Score · {nombre}"] = scores[doc_id]

    filas_posiciones.append(fila)

print(f"Consulta: {consulta}")
print(f"Longitud media de la colección: {longitud_media:,.1f} tokens")
display(pd.DataFrame(filas_posiciones).round(6))
filas_aportes = []

for doc_id in elegidos:
    for t in terminos:
        fila = {
            "doc_id": doc_id,
            "Término": t,
            "f(t,d)": frecuencias[doc_id][t],
            "df(t)": df_corpus.get(t, 0),
        }

        for nombre, (_, _, aportes) in resultados.items():
            fila[f"Aporte · {nombre}"] = aportes[doc_id][t]

        filas_aportes.append(fila)

display(pd.DataFrame(filas_aportes).round(6))

Consulta: whale ship captain
Longitud media de la colección: 72,554.4 tokens


,doc_id,|d|,Posición · Base,Score · Base,Posición · Sin ajuste por longitud,Score · Sin ajuste por longitud,Posición · Mayor efecto de las repeticiones,Score · Mayor efecto de las repeticiones
0,10469,3444,109,3.194042,283,1.949413,100,3.722143
1,100,991578,74,3.898865,5,6.114729,69,4.453984
2,10545,175209,1,6.422759,1,6.534782,1,8.571045


,doc_id,Término,"f(t,d)",df(t),Aporte · Base,Aporte · Sin ajuste por longitud,Aporte · Mayor efecto de las repeticiones
0,10469,whale,1,142,3.194042,1.949413,3.722143
1,10469,ship,0,623,0.000000,0.000000,0.000000
2,10469,captain,0,550,0.000000,0.000000,0.000000
3,100,whale,9,142,1.786960,3.784154,1.754469
4,100,ship,68,623,0.878679,1.023433,1.085110
5,100,captain,189,550,1.233226,1.307142,1.614405
6,10545,whale,84,142,4.166039,4.228304,5.574662
7,10545,ship,28,623,0.956960,0.998692,1.237960
8,10545,captain,205,550,1.299759,1.307786,1.758424


### Declaración de uso de IA
Durante la realización de este trabajo se han utilizado herramientas de inteligencia artificial con el objetivo de comprender con mayor profundidad la materia y conceptos necesarios para realizar la actividad, mejorar el código redactado, explicar y realizar pseudocódigo, investigar acerca de librerías y redacción en Python.